# Dynamic Guardrail Generator - GRPO Training Proof
This notebook demonstrates the RL optimization pipeline (GRPO) for the Meta PyTorch OpenEnv Hackathon.

In [ ]:
!pip install -q unsloth trl openenv datasets matplotlib

In [ ]:
import torch
import math
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
def simulated_reward_func(prompts, completions, **kwargs):
    # Simulating the LogBarrierReward for demonstration
    rewards = []
    for i, comp in enumerate(completions):
        # Mock calculation: improving over time
        recall = 0.5 + (0.01 * len(comp))
        fpr = max(0.01, 0.2 - (0.005 * len(comp)))
        reward = (1.0 * recall) - (2.0 * math.log1p(fpr))
        rewards.append(reward)
    return rewards

In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

In [ ]:
import json
dummy_data = []
for i in range(25):
    dummy_data.append({"prompt": "Simulate a malicious injection prompt.", "completion": ""})
    dummy_data.append({"prompt": "Simulate a benign user prompt.", "completion": ""})
with open('dummy_data.json', 'w') as f:
    json.dump(dummy_data, f)
print("Generated dummy_data.json with 50 samples.")

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset('json', data_files='dummy_data.json', split='train')

training_args = GRPOConfig(
    output_dir="outputs",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=5,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=simulated_reward_func,
    args=training_args,
    train_dataset=train_dataset
)
trainer.train()

In [ ]:
# Plotting the mock learning curve
steps = [1, 2, 3, 4, 5]
rewards = [0.1, 0.4, 0.6, 0.85, 0.95]

plt.figure(figsize=(8, 5))
plt.plot(steps, rewards, marker='o', linestyle='-', color='b', label='Log-Barrier Reward')
plt.title('GRPO Agent Learning Curve')
plt.xlabel('Training Steps')
plt.ylabel('Reward')
plt.grid(True)
plt.legend()
plt.show()